In [1]:
import numpy as np
import pandas as pd
import geopandas as gp

from mgwr.gwr import MGWR as OfficialMGWR
from mgwr.sel_bw import Sel_BW

from src.kernel.gwr_kernel import GwrKernel
from src.model.gwr import GWR
from src.model.mgwr import MGWR
from src.dataset.interfaces.idataset import FieldInfo
from src.dataset.spatial_dataset import SpatialDataset

Create the Georgia dataset

In [3]:
georgia_data = pd.read_csv(r'../../../data/GData_utm.csv')
# georgia_geometry = gp.read_file(r'../../../data/G_utm.shp')
dataset = SpatialDataset(
    georgia_data,
    FieldInfo(
        predictor_fields=['PctFB', 'PctBlack', 'PctRural'],
        response_field='PctBach',
        coordinate_x_field='X',
        coordinate_y_field='Y'
    ),
    isSpherical=False,
    # geometry=georgia_geometry
)
# dataset.plot_map()

SpatialDataset : Data schema is verified.


Create and fit the MGWR model (Official MGWR codebase)

In [4]:
g_X = dataset.X[:, 1:]
g_y = dataset.y.reshape(-1, 1)
g_coords = dataset.coordinates.tolist()

mgwr_selector = Sel_BW(g_coords, g_y, g_X, multi=True)
official_mgwr_bw = mgwr_selector.search(multi_bw_min=[2])
official_mgwr_results = OfficialMGWR(g_coords, g_y, g_X, mgwr_selector, hat_matrix=True).exact_fit()
print(official_mgwr_bw)

Backfitting:   0%|          | 0/200 [00:00<?, ?it/s]

[ 92. 101. 136. 158.]


In [ ]:
official_mgwr_results.summary()

Model type                                                         Gaussian
Number of observations:                                                 159
Number of covariates:                                                     4

Global Regression Results
---------------------------------------------------------------------------
Residual sum of squares:                                             71.793
Log-likelihood:                                                    -162.399
AIC:                                                                332.798
AICc:                                                               335.191
BIC:                                                               -713.887
R2:                                                                   0.548
Adj. R2:                                                              0.540

Variable                              Est.         SE  t(Est/SE)    p-value
------------------------------- ---------- ---------- ------

Create the MGWR model (lmgwr codebase)

In [5]:
kernel = GwrKernel(dataset, 'bisquare')
refactored_mgwr = MGWR(dataset, kernel)
refactored_mgwr.update_bandwidth_set([92., 101., 136., 158.]).exact_fit()

MGWR : MGWR model is initialized.
GWR : GWR model is initialized.
GWR : GWR model is initialized.
GWR : GWR model is initialized.
GWR : GWR model is initialized.
MGWR : MGWR model fitting is complete.


Compare outputs

In [10]:

s_matrix_match = np.allclose(refactored_mgwr.S, official_mgwr_results.S)
betas_match = np.allclose(refactored_mgwr.betas, official_mgwr_results.params)

print("="*60)
print("MGWR Model Comparison")
print("="*60)
print(f"{'Metric':30} {'Custom MGWR':>12} {'Official MGWR':>15}")
print("-"*60)
print(f"{'R-squared':30} {refactored_mgwr.r_squared:12.4f} {official_mgwr_results.R2:15.4f}")
print(f"{'AIC':30} {refactored_mgwr.aic:12.2f} {official_mgwr_results.aic:15.2f}")
print(f"{'AICc':30} {refactored_mgwr.aicc:12.2f} {official_mgwr_results.aicc:15.2f}")
print("-"*60)
print(f"{'Coefficient matrices match?':30} {str(betas_match):>12}")
print(f"{'S matrices match?':30} {str(s_matrix_match):>12}")
print("="*60)


MGWR Model Comparison
Metric                          Custom MGWR   Official MGWR
------------------------------------------------------------
R-squared                            0.6799          0.6799
AIC                                  294.85          294.85
AICc                                 297.12          297.12
------------------------------------------------------------
Coefficient matrices match?            True
S matrices match?                      True
